# 未知語 ([UNK]) がどれくらい出るか調べる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/07_subword/unk_rate.ipynb)

NLP クックブック 07-11 ／ サブワードとトークナイザ

使おうとしているモデルのトークナイザが、手元のテキストをきちんと表現できているか確認したい。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
from transformers import AutoTokenizer

texts = ["𠮷野家で牛丼を食べた😋", "髙橋さんと﨑山さんが来た。", "한국어와 日本語 and English", "ｷﾀ━━━(ﾟ∀ﾟ)━━━!!", "普通の日本語の文です。"]

for name in ["tohoku-nlp/bert-base-japanese-v3", "bert-base-multilingual-cased", "xlm-roberta-base"]:
    tok = AutoTokenizer.from_pretrained(name)
    total, unk = 0, 0
    for t in texts:
        ids = tok(t, add_special_tokens=False)["input_ids"]
        total += len(ids)
        unk += sum(i == tok.unk_token_id for i in ids)
    print(f"{name:36} UNK 率 {unk / total:.1%} ({unk}/{total})")
    print("   ", tok.tokenize(texts[0]))

tohoku-nlp/bert-base-japanese-v3     UNK 率 9.8% (5/51)
    ['[UNK]', '野', '##家', 'で', '牛', '##丼', 'を', '食べ', 'た', '[UNK]']
bert-base-multilingual-cased         UNK 率 8.3% (4/48)
    ['[UNK]', '野', '家', 'で', '牛', '丼', 'を', '食', '[UNK]']
xlm-roberta-base                     UNK 率 4.3% (2/47)
    ['▁', '𠮷', '野', '家', 'で', '牛', '丼', 'を', '食べた', '😋']


- WordPiece は語彙にない文字を含む単語をまるごと `[UNK]` にする。SentencePiece 系 (XLM-R) は文字単位にフォールバックするので UNK が出にくい。
- UNK が多い場合は、NFKC 正規化や異体字の置換などの前処理を検討する。